# Practical 09 — Financial NLP and Sentiment Analysis

This notebook runs the Chapter 9 sentiment pipeline **end to end on the bundled,
fictional NovaCorp headlines**, fully offline. A deterministic finance lexicon scores
each headline's net polarity (with negation and intensifier handling), then the scores
are aggregated by date into a daily bullish / bearish / neutral signal.

In the course project you drive these same tools from an agent in Claude Code / Cline.
Here, the Colab-friendly counterpart, we call the **reference tools directly** so you
can watch every score and aggregation happen and read its real output.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "09-financial-nlp-sentiment"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Look at the raw inputs

Two bundled files drive everything, with no network access:

* `data/headlines.csv` — nine dated, fictional headlines for **NovaCorp Inc.** across
  three days (engineered as a bullish, a bearish, and a neutral day).
* `data/lexicon.json` — a small Loughran-McDonald-style polarity lexicon: positive and
  negative word lists, `negators` that flip a sign, and `intensifiers` that scale it.

In [ ]:
import json
from tools._common import load_headlines, load_lexicon

headlines = load_headlines()
print(f"{len(headlines)} headlines loaded\n")
for h in headlines:
    print(f"  {h['date']}  {h['id']}  {h['text']}")

lex = load_lexicon()
print("\nLexicon sizes:",
      {k: len(v) for k, v in lex.items()})
print("Sample positive words:", sorted(lex["positive"])[:6])
print("Sample negative words:", sorted(lex["negative"])[:6])
print("Negators:", sorted(lex["negators"]))
print("Intensifiers:", lex["intensifiers"])

## Step 1 — Score one headline

`tools.lexicon.score(text)` is the core scorer. It tokenizes the text, gives each
polarity word +1 / -1, flips that sign if a negator sits in the prior three tokens,
and amplifies it if an intensifier sits in the prior two tokens. The **net polarity**
is the signed sum divided by the number of sentiment-bearing words, so it lands in
roughly [-1.5, 1.5]. It also returns which words it matched.

In [ ]:
from tools.lexicon import score

example = "NovaCorp beats earnings estimates as cloud revenue hits record growth"
result = score(example)
print(json.dumps(result, indent=2))

Two positive words (`beats`, `record`, `growth` — three here) and no negatives, so the
net polarity is positive. Now a clearly bearish headline, with intensifier handling.

In [ ]:
bearish = score("NovaCorp posts a steep loss amid a fraud investigation")
print(json.dumps(bearish, indent=2))
# `steep` is an intensifier (x1.5) sitting before `loss`, so that word weighs -1.5.

## Step 2 — "Things to try": negation flips a score

A bag-of-words lexicon would call any sentence with `beat` positive. This scorer scans
the three tokens before each polarity word for a negator, so "does not beat" flips to
negative. This is exactly the `2024-02-14` case (`h007`) that makes that day net neutral.

In [ ]:
plain   = score("NovaCorp did beat revenue expectations")
negated = score("NovaCorp does not beat revenue expectations citing soft demand")
print(f"plain   polarity = {plain['polarity']:+.4f}   matched+={plain['positive']}")
print(f"negated polarity = {negated['polarity']:+.4f}   matched-={negated['negative']}")
assert plain["polarity"] > 0 and negated["polarity"] < 0
print("=> the same word 'beat' flips sign because 'not' precedes it within 3 tokens.")

Intensifiers work the same way: a word in the two tokens before a polarity word scales
its magnitude, so "strong gains" reads as more bullish than plain "gains".

In [ ]:
g_plain = score("NovaCorp reported gains")
g_int   = score("NovaCorp reported strong gains")
print(f"'gains'        -> {g_plain['polarity']:+.4f}")
print(f"'strong gains' -> {g_int['polarity']:+.4f}  (intensifier x1.5)")

## Step 3 — Aggregate all headlines into a daily signal

`tools.aggregate.build_signal()` scores every bundled headline, groups by date, and
takes the NumPy mean polarity per day. Each day is labelled against a symmetric
threshold (default **0.15**): `mean > 0.15` bullish, `mean < -0.15` bearish, else neutral.

In [ ]:
from tools.aggregate import build_signal, daily_signal, score_records, label, DEFAULT_THRESHOLD

signal = build_signal()
print(f"threshold = {DEFAULT_THRESHOLD}\n")
print(json.dumps(signal["by_date"], indent=2))

A compact view of the three days, and the per-headline detail behind them.

In [ ]:
print(f"{'date':<12}{'label':<9}{'mean':>8}{'count':>7}   item_ids")
for d, info in signal["by_date"].items():
    print(f"{d:<12}{info['label']:<9}{info['mean']:>8.4f}{info['count']:>7}   {info['item_ids']}")

print("\nper-headline polarities:")
for h in signal["headlines"]:
    print(f"  {h['date']}  {h['id']}  {h['polarity']:+.4f}  {h['text']}")

Note `2024-02-14`: `h007` ("does not beat ...") is negative and `h008` ("launches a new
analytics platform") and `h009` are near zero, so the day's mean falls inside the ±0.15
band and lands **neutral** — the negation is respected all the way through the signal.

In [ ]:
neutral_day = {d: v for d, v in signal["by_date"].items() if v["label"] == "neutral"}
print("neutral day(s):", json.dumps(neutral_day, indent=2))

## Step 4 — Aggregate a single date

You can score just one day, exactly like `python -m tools.aggregate --date 2024-02-13`.

In [ ]:
bearish_records = [h for h in load_headlines() if h["date"] == "2024-02-13"]
bearish_scored = score_records(bearish_records)
bearish_signal = daily_signal(bearish_scored)
print(json.dumps(bearish_signal, indent=2))

## Step 5 — "Things to try": a labelling cutoff manufactures signal

The 0.15 threshold is a modelling choice, not a fact. The three bundled days have
strong means (1.08, -0.78, 0.0), so they are robust to it. But a *borderline* day —
one mild positive headline among neutral ones — has a small mean that sits right on the
cutoff: drop the threshold toward 0 and it flips from neutral to bullish, while the mean
itself never changes. That is signal created by the cutoff, not by the news.

In [ ]:
# One mild positive headline diluted by several neutral ones -> a small, borderline mean.
borderline = [{"date": "2024-03-01", "id": "b1", "text": "NovaCorp reported a modest profit"}]
borderline += [
    {"date": "2024-03-01", "id": f"b{i}", "text": t}
    for i, t in enumerate([
        "NovaCorp held its annual product briefing",
        "NovaCorp confirmed its head office relocation",
        "NovaCorp scheduled its next earnings call",
        "NovaCorp updated its corporate logo",
        "NovaCorp published its sustainability report",
    ], start=2)
]
b_mean = daily_signal(score_records(borderline))["2024-03-01"]["mean"]
print(f"borderline day mean = {b_mean:+.4f}  (one mild positive diluted by five neutral)\n")
print(f"{'threshold':>10} -> label")
for thr in (0.05, 0.15, 0.30):
    print(f"{thr:>10.2f} -> {label(b_mean, thr)}")
print("\nThe mean is fixed; only the verdict moves as the cutoff moves.")

## Step 6 — Where a bag-of-words lexicon misreads

The scorer has no notion of sarcasm, hedging, or context. A hedged or ironic headline
can score with the wrong sign — a built-in limitation worth seeing first-hand.

In [ ]:
for text in [
    "NovaCorp's so-called record growth is anything but impressive",  # sarcasm reads positive
    "NovaCorp may avoid a loss if demand does not weaken further",     # hedged / double negation
]:
    s = score(text)
    print(f"polarity {s['polarity']:+.4f}  +{s['positive']} -{s['negative']}")
    print(f"   {text}\n")

## Recap / next steps

We ran the full Chapter 9 pipeline directly on the bundled data:

1. **Score** a headline's net polarity with `tools.lexicon.score` (negation + intensity).
2. **Aggregate** all headlines into a daily mean and bullish/bearish/neutral label with
   `tools.aggregate.build_signal` / `daily_signal`.
3. **Inspect** the signal: the three NovaCorp days come out bullish, bearish, neutral.

Things to try next: add a headline to `data/headlines.csv` on an existing date and re-run
the aggregation; move a word like "guidance" into the lexicon's positive list and see how
many headlines change sign; sweep the threshold further; or run `python -m pytest -q` to
confirm the tools still pass offline. In the agentic project, an agent chooses inputs and
writes a cited summary — but every number still comes from these deterministic tools.